# EP01 - Oblique Random Forest

- Enzo Koichi Jojima 14568285
- Pedro Biagioni Matusita

## Imports

In [25]:
import numpy as np
from typing import Literal, TypedDict
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
import warnings
from sklearn.exceptions import ConvergenceWarning
from itertools import combinations
from scipy import stats
from sklearn.model_selection import train_test_split
import pandas as pd
from datetime import datetime
import time
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from multiprocessing import Pool
import os
from sklearn.model_selection import KFold

cores = os.cpu_count()

## Helper Funcitions

In [17]:
def loadFile(path: str):
    data = np.load(path)

    X_train = data['X_train']
    Y_train = data['y_train']
    X_test = data['X_test']


    print(25*"=" + f"\nLoaded {path}")
    return X_train, Y_train, X_test

def splitData(
    X: np.ndarray,
    Y: np.ndarray,
    testSize: float = 0.25,
    randomState: int = 42,
    shuffle: bool = True
):
    # estratifica para preservar a proporção das 3 classes (desbalanceadas)
    # no train e no test locais — só faz sentido com shuffle=True
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y,
        test_size=testSize,
        random_state=randomState,
        shuffle=shuffle,
        stratify=Y if shuffle else None
    )
    return X_train, X_test, y_train, y_test

def exportPredictions(
    y_test: np.ndarray
):
    dataframe = pd.DataFrame({
        'ID': np.arange(1, len(y_test) + 1),
        'Prediction': y_test
    })

    now = datetime.now().isoformat()

    print(now)

    dataframe.to_csv(f"outputs/predictions_{now}.csv", index=False)

def accuracyAndError(
    y_hat: np.ndarray, # predictions
    y_exp: np.ndarray  # expected
):
    # calculating error rate
    hits = 0

    for index, y in enumerate(y_hat):
        if (y == y_exp[index]):
            hits += 1

    accuracy = hits / len(y_exp)
    errorRate = 1 - accuracy

    return accuracy, errorRate

def _evaluate(name, model, x_train, y_train, x_test, y_test, **fit_kwargs):
    t0 = time.perf_counter()
    model.fit(x_train, y_train, **fit_kwargs)
    t_fit = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_hat = model.predict(x_test)
    t_pred = time.perf_counter() - t0

    acc, _ = accuracyAndError(y_hat, y_test)
    print(f"{name:36s} acc={acc:.4f}  fit={t_fit:6.2f}s  pred={t_pred:5.2f}s")
    return acc


## Classes

### Node Class

In [18]:
class Node():
    def __init__(self,
                 w_star=None,       # vetor normal (ortogonal/PCA/SVM linear)
                 th_star=None,
                 left=None,
                 right=None,
                 label=None,
                 svm_model=None,    # objeto SVC (SVM com kernel)
                 scaler=None):      # StandardScaler do nó

        # for decision node
        self.w_star = w_star
        self.th_star = th_star
        self.left = left
        self.right = right

        # y_hat for leaf nodes
        self.label = label

        # para SVM com kernel
        self.svm_model = svm_model
        self.scaler = scaler


### JojiTree Class

In [19]:
from node import Node

import numpy as np
from typing import Literal, TypedDict
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
import warnings
from sklearn.exceptions import ConvergenceWarning
from itertools import combinations
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

class OrthogonalParams(TypedDict):
    type: str = 'orthogonal'

class PCAParams(TypedDict):
    type: str = 'PCA'
    c: int

class SVMParams(TypedDict):
    type: str = 'SVM'
    C: float
    kernel: str  # 'linear', 'rbf', 'poly', 'sigmoid'



type GainMethod = Literal["entropy", "gini"]
type SplitMethod = OrthogonalParams | PCAParams | SVMParams
type NDArray = np.ndarray

GAINMETHODS = ['entropy', 'gini']

class JojiTree:
    def __init__(
        self,
        maxDepth: int = 4,
        splitMethod: SplitMethod = OrthogonalParams(), # random, PCA, SVM, ...
        gainMethod: GainMethod  = "gini", # entropy, ...,
        verbose: bool = False
    ):
        if splitMethod['type'] == 'PCA' and splitMethod['c'] <= 0:
            raise ValueError("c must be a positive integer.")

        if (not GAINMETHODS.__contains__(gainMethod)):
            raise TypeError(f'Invalid Gain Method: {gainMethod}.')

        self.root = None
        self.maxDepth = maxDepth
        self.gainMethod = gainMethod
        self.splitMethod = splitMethod
        self.originalFeatureIndexes = None

        if (verbose):
            print(18*"=" + f" Created JojiTree " + 18*"=")
            print(f'gainMethod: {self.gainMethod}')
            print(f'splitMethod: {self.splitMethod}')
            print(f'maxDepth: {self.maxDepth}')

    def _buildTree(self, X: NDArray, Y: NDArray, depth: int = 0) -> Node:
        # Critério de parada MaxDepth
        if depth >= self.maxDepth:
            return Node(label=self._calculateLeafLabel(Y))

        # Critério de parada #1: quando nesse ramo há apenas 1 classe
        if len(np.unique(Y)) == 1:
            return Node(label=Y[0]) # O Leaf Label é aquela classe

        X_left, X_right, Y_left, Y_right, w_star, th_star, svm_model, scaler = self._getBestSplit(X, Y)

        if (X_left is None or X_right is None or Y_left is None or Y_right is None):
            return Node(label=self._calculateLeafLabel(Y))

        if len(X_left) == 0 or len(X_right) == 0:
            return Node(label=self._calculateLeafLabel(Y))

        Xi_left, Yi_left = np.array(X_left), np.array(Y_left)
        Xi_right, Yi_right = np.array(X_right), np.array(Y_right)

        left_subtree = self._buildTree(Xi_left, Yi_left, depth=depth+1)
        right_subtree = self._buildTree(Xi_right, Yi_right, depth=depth+1)

        return Node(w_star, th_star, left_subtree, right_subtree,
                    svm_model=svm_model, scaler=scaler)

    def _informationGain(self, parent: NDArray, l_child: NDArray, r_child: NDArray) -> float:
        match self.gainMethod:
            case "entropy":

                entParent = self._entropyCalc(parent)
                entLChild = self._entropyCalc(l_child)
                entRChild = self._entropyCalc(r_child)

                nu = len(l_child) / len(parent)

                return entParent - ((nu * entLChild )+ ((1-nu)*entRChild))

            case "gini":
                giniParent = self._giniCalc(parent)
                giniLChild = self._giniCalc(l_child)
                giniRChild = self._giniCalc(r_child)

                nu = len(l_child) / len(parent)

                return giniParent - ((nu * giniLChild) + ((1-nu) * giniRChild))

        return -1

    def _entropyCalc(self, y: NDArray):
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return entropy(p, base=2)

    def _giniCalc(self, y: NDArray) -> float:
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return 1 - np.sum(p ** 2)

    def _calculateLeafLabel(self, Y: NDArray) -> NDArray:
        values, counts = np.unique(Y, return_counts=True)
        return values[np.argmax(counts)]

    def fit(self, X: NDArray, Y: NDArray, lda_components: int | None = None):

        self.lda = None
        if lda_components is not None:
            n_classes = len(np.unique(Y))
            n_components = min(lda_components, n_classes - 1)
            self.lda = LinearDiscriminantAnalysis(n_components=n_components)
            X = self.lda.fit_transform(X, Y.ravel())

        self.root = self._buildTree(X, Y)

    def predict(self, X: np.ndarray) -> list:
        if self.lda is not None:
            X = self.lda.transform(X)

        predictions = []
        for x in X:
            y_hat = self._makePrediction(x, self.root)
            predictions.append(y_hat)

        return predictions

    def _makePrediction(self, x: NDArray, tree: Node):
        if (self.splitMethod['type'] == 'orthogonal'):
            return self._makeOrthogonalPrediction(x, tree)
        else:
            return self._makeObliquePrediction(x, tree)

    def _makeOrthogonalPrediction(self, x: NDArray, tree: Node):
        if tree.label is not None: #Leaf
            return tree.label

        # para o caso ortogonal, w_star é o índice da feature_star
        feature_val = x[tree.w_star]
        if feature_val<=tree.th_star:
            return self._makeOrthogonalPrediction(x, tree.left)
        else:
            return self._makeOrthogonalPrediction(x, tree.right)

    def _makeObliquePrediction(self, x: NDArray, tree: Node):
        if tree.label is not None:
            return tree.label

        if tree.svm_model is not None:
            # SVM com kernel: usa decision_function
            x_scaled = tree.scaler.transform(x.reshape(1, -1))
            projection = tree.svm_model.decision_function(x_scaled)[0]
        else:
            # PCA ou SVM linear: produto interno direto
            projection = tree.w_star @ x

        if projection <= tree.th_star:
            return self._makeObliquePrediction(x, tree.left)
        else:
            return self._makeObliquePrediction(x, tree.right)

    def _getBestSplit(self, X, Y):
        svm_model, scaler = None, None
        match self.splitMethod['type']:
            case "orthogonal":
                X_left, X_right, Y_left, Y_right, w_star, th_star = self._getOrthogonalSplits(X, Y)
            case "PCA":
                X_left, X_right, Y_left, Y_right, w_star, th_star = self._getPCASplits(X, Y)
            case "SVM":
                X_left, X_right, Y_left, Y_right, w_star, th_star, svm_model, scaler = self._getSVMSplits(X, Y)

        return X_left, X_right, Y_left, Y_right, w_star, th_star, svm_model, scaler


    def _getOrthogonalSplits(self, X, Y):
        w_star, th_star = -1, -1
        max_info_gain = -float("inf")
        X_left_best = X_right_best = Y_left_best = Y_right_best = None

        for feature_i in range(X.shape[-1]):
            feature_values = X[:, feature_i]
            th, gain = self._bestThresholdForProjection(feature_values, Y)  # reusa a função vetorizada

            if th is not None and gain > max_info_gain:
                max_info_gain = gain
                w_star, th_star = feature_i, th
                mask = feature_values <= th
                X_left_best, X_right_best = X[mask], X[~mask]
                Y_left_best, Y_right_best = Y[mask], Y[~mask]

        return X_left_best, X_right_best, Y_left_best, Y_right_best, w_star, th_star

    def _getPCASplits(self, X: np.ndarray, Y: np.ndarray):
        w_star, th_star = None, -1
        max_info_gain = -float("inf")
        X_left_best = X_right_best = Y_left_best = Y_right_best = None

        local_mean = X.mean(axis=0)
        Xc = X - local_mean
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

        n_components = min(self.splitMethod['c'], Vt.shape[0])
        X_proj = U[:, :n_components] * S[:n_components]

        for index, projections in enumerate(X_proj.T):
            th, gain = self._bestThresholdForProjection(projections, Y)

            if th is not None and gain > max_info_gain:
                max_info_gain = gain
                w_star = Vt[index]
                th_star = th + (local_mean @ Vt[index])  # ajusta para o espaço não-centralizado

                mask = projections <= th
                X_left_best, X_right_best = X[mask], X[~mask]
                Y_left_best, Y_right_best = Y[mask], Y[~mask]

        return X_left_best, X_right_best, Y_left_best, Y_right_best, w_star, th_star

    def _bestThresholdForProjection(self, projections: NDArray, Y: NDArray):
        """Dado z = X @ w, encontra o threshold que maximiza o ganho.

        Vetorizado: ordena uma vez e varre apenas os pontos médios entre
        valores consecutivos (candidatos ótimos), sem montar listas por amostra.
        Retorna (best_th, best_gain) ou (None, -inf) se nenhum split válido.
        """
        order = np.argsort(projections, kind="mergesort")
        z = projections[order]

        # Candidatos: pontos médios onde z muda de valor (split válido com ambos os lados não-vazios)
        change = np.where(z[1:] != z[:-1])[0]
        if change.size == 0:
            return None, -float("inf")

        thresholds = (z[change] + z[change + 1]) / 2.0

        best_th, best_gain = None, -float("inf")
        for th in thresholds:
            mask = projections <= th
            gain = self._informationGain(Y, Y[mask], Y[~mask])
            if gain > best_gain:
                best_gain = gain
                best_th = th

        return best_th, best_gain

    def _splitByProjection(self, X: NDArray, Y: NDArray, w: NDArray, th: float):
        z = X @ w
        mask = z <= th
        return X[mask], X[~mask], Y[mask], Y[~mask]

    def _getSVMSplits(self, X: NDArray, Y: NDArray):
        classes = np.unique(Y)
        C = self.splitMethod.get('C', 1.0)
        kernel = self.splitMethod.get('kernel', 'linear')

        X_left_best = X_right_best = Y_left_best = Y_right_best = None
        w_star, th_star = None, -1
        svm_best, scaler_best = None, None
        max_info_gain = -float("inf")

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        partitions = []
        k = len(classes)
        for r in range(1, k // 2 + 1):
            for group in combinations(classes, r):
                if r == k - r and classes[0] not in group:
                    continue
                partitions.append(set(group))

        for group in partitions:
            binary_y = np.array([0 if c in group else 1 for c in Y])

            if len(np.unique(binary_y)) < 2:
                continue

            try:
                if kernel == 'linear':
                    svm = LinearSVC(C=C, dual='auto', max_iter=2000)
                    with warnings.catch_warnings():
                        warnings.filterwarnings('ignore', category=ConvergenceWarning)
                        svm.fit(X_scaled, binary_y)
                    projections = svm.decision_function(X_scaled)
                else:
                    svm = SVC(kernel=kernel, C=C, max_iter=-1)
                    svm.fit(X_scaled, binary_y)
                    projections = svm.decision_function(X_scaled)
            except Exception:
                continue

            th, gain = self._bestThresholdForProjection(projections, Y)
            if th is None:
                continue

            if gain > max_info_gain:
                max_info_gain = gain
                th_star = th

                mask = projections <= th
                X_left_best = X[mask]
                X_right_best = X[~mask]
                Y_left_best = Y[mask]
                Y_right_best = Y[~mask]

                if kernel == 'linear':
                    # LinearSVC: extrai w diretamente, não precisa guardar modelo
                    w = svm.coef_[0] / scaler.scale_
                    w_star = w / np.linalg.norm(w)
                    svm_best = None
                    scaler_best = None
                else:
                    # Kernel não-linear: guarda modelo e scaler para predição
                    w_star = None
                    svm_best = svm
                    scaler_best = scaler

        return X_left_best, X_right_best, Y_left_best, Y_right_best, w_star, th_star, svm_best, scaler_best


### JojiForest Class

In [20]:
def _trainTree(args):
    X_bag, Y_bag, subsamplingCols, maxDepth, splitMethod, gainMethod, lda_components = args
    tree = JojiTree(
        maxDepth=maxDepth,
        splitMethod=splitMethod,
        gainMethod=gainMethod
    )
    tree.fit(X_bag, Y_bag, lda_components=lda_components)
    tree.originalFeatureIndexes = subsamplingCols
    return tree

def _predictTree(args):
    tree, X_test = args
    return tree.predict(X_test[:, tree.originalFeatureIndexes])

# Random Forest
class JojiForest:

    def __init__(
        self,
        featuresPerTree: int,
        samplesPerTree: int,
        repeatedSampling: bool = True,
        treeCount: int = 5,
        maxDepth: int = 4,
        gainMethod: GainMethod = "entropy",
        splitMethod: SplitMethod = OrthogonalParams(),
        lda_components: int | None = None,
        n_jobs: int = -1,  # -1 = todos os núcleos
    ):
        if featuresPerTree <= 0:
            raise ValueError("featurePerTree must be a positive integer.")

        if not GAINMETHODS.__contains__(gainMethod):
            raise TypeError(f'Invalid Gain Method: {gainMethod}.')

        self.featuresPerTree = featuresPerTree
        self.samplesPerTree = samplesPerTree
        self.repeatedSampling = repeatedSampling
        self.treeCount = treeCount
        self.maxDepth = maxDepth
        self.gainMethod = gainMethod
        self.splitMethod = splitMethod
        self.lda_components = lda_components
        self.n_jobs = n_jobs
        self.trees: list[JojiTree] = []

    def fit(self, X_train: NDArray, Y_train: NDArray):
        n, m = X_train.shape

        if self.samplesPerTree > n:
            raise ValueError(f"X_train must have at least {self.samplesPerTree} samples.")

        if self.featuresPerTree > m:
            raise ValueError(f"X_train must have at least {self.featuresPerTree} features.")

        rng = np.random.default_rng()

        args = []
        for _ in range(self.treeCount):
            baggingIndexes = rng.choice(n, size=self.samplesPerTree, replace=self.repeatedSampling)
            subsamplingCols = rng.choice(m, size=self.featuresPerTree, replace=False)

            X_bag = X_train[baggingIndexes][:, subsamplingCols]
            Y_bag = Y_train[baggingIndexes]

            args.append((X_bag, Y_bag, subsamplingCols,
                         self.maxDepth, self.splitMethod,
                         self.gainMethod, self.lda_components))

        n_jobs = self.n_jobs if self.n_jobs > 0 else None  # None = todos os núcleos
        with Pool(n_jobs) as pool:
            self.trees = pool.map(_trainTree, args)

    def predict(self, X_test: NDArray):
        args = [(tree, X_test) for tree in self.trees]

        n_jobs = self.n_jobs if self.n_jobs > 0 else None
        with Pool(n_jobs) as pool:
            results = pool.map(_predictTree, args)

        predictions = np.array(results)  # (treeCount, n_test)
        return stats.mode(predictions, axis=0, keepdims=True).mode.flatten()


## Testing Parameters

### Testing Number of Trees Per Forest

In [ ]:
def testTreeNumber():
    X_train, Y_train, X_test = loadFile('data.npz')
    x_train, x_test, y_train, y_test = splitData(X_train, Y_train)
    n_train, m = x_train.shape

    for k in [10, 20, 50, 100, 200]:
        forest = JojiForest(
                    featuresPerTree=25,
                    samplesPerTree=int(n_train * 0.8),
                    repeatedSampling=True,
                    treeCount=k,
                    maxDepth=4,
                    gainMethod='gini',
                    splitMethod=SVMParams({'type': 'SVM', 'C': 1.0, 'kernel': 'sigmoid'}),
                    lda_components=2,
                    n_jobs=cores-1
                )
        forest.fit(x_train, y_train)
        acc, _ = accuracyAndError(forest.predict(x_test), y_test)
        print(f"k={k}: {acc:.4f}")

if __name__ == '__main__':
    testTreeNumber()

Loaded data.npz
k=10: 0.7288
k=20: 0.7246
k=50: 0.7246
k=100: 0.7331
k=200: 0.7345


### Testing Features Per Tree

m=34, variações de featuresPerTree: [5, 10, 17, 23, 30, 34]

============================== orthogonal ==============================

Forest[f= 5]                         acc=0.6540  fit= 89.06s  pred= 0.13s

Forest[f=10]                         acc=0.7147  fit=172.56s  pred= 0.14s

Forest[f=17]                         acc=0.7486  fit=188.55s  pred= 0.13s

Forest[f=23]                         acc=0.7542  fit=188.20s  pred= 0.20s

Forest[f=30]                         acc=0.7542  fit=202.77s  pred= 0.14s

Forest[f=34]                         acc=0.7458  fit=212.42s  pred= 0.17s

============================== SVM-linear ==============================

Forest[f= 5]                         acc=0.6921  fit= 26.50s  pred= 0.30s

Forest[f=10]                         acc=0.7119  fit= 47.00s  pred= 0.30s

Forest[f=17]                         acc=0.7246  fit= 52.23s  pred= 0.27s

Forest[f=23]                         acc=0.7302  fit= 57.27s  pred= 0.24s

Forest[f=30]                         acc=0.7500  fit= 65.28s  pred= 0.31s

Forest[f=34]                         acc=0.7429  fit= 75.00s  pred= 0.30s

============================== SVM-sigmoid ==============================

Forest[f= 5]                         acc=0.5876  fit= 51.65s  pred=31.90s

Forest[f=10]                         acc=0.6893  fit= 68.83s  pred=31.97s

Forest[f=17]                         acc=0.7232  fit= 71.45s  pred=31.54s

Forest[f=23]                         acc=0.7316  fit= 73.86s  pred=31.23s

Forest[f=30]                         acc=0.7401  fit= 83.20s  pred=31.09s

Forest[f=34]                         acc=0.7387  fit= 92.20s  pred=30.99s

============================== SVM-rbf ==============================

Forest[f= 5]                         acc=0.6554  fit= 50.69s  pred=32.20s

Forest[f=10]                         acc=0.7090  fit= 68.69s  pred=31.80s

Forest[f=17]                         acc=0.7401  fit= 71.02s  pred=31.77s

Forest[f=23]                         acc=0.7458  fit= 75.69s  pred=31.70s

Forest[f=30]                         acc=0.7500  fit= 82.30s  pred=31.50s

Forest[f=34]                         acc=0.7514  fit= 90.09s  pred=31.70s

============================== SVM-linear ==============================

Forest[f= 5]                         acc=0.6794  fit= 23.61s  pred= 0.32s

Forest[f=10]                         acc=0.7161  fit= 45.86s  pred= 0.31s

Forest[f=17]                         acc=0.7246  fit= 52.38s  pred= 0.31s

Forest[f=23]                         acc=0.7401  fit= 57.35s  pred= 0.34s

Forest[f=30]                         acc=0.7415  fit= 67.54s  pred= 0.28s

Forest[f=34]                         acc=0.7415  fit= 72.22s  pred= 0.28s

In [23]:
def featuresPerTree():
    X_train, Y_train, X_test = loadFile('data.npz')
    x_train, x_test, y_train, y_test = splitData(X_train, Y_train)
    n_train, m = x_train.shape

    splitMethods = {
        'orthogonal': OrthogonalParams({'type': 'orthogonal'}),
        'PCA-1': PCAParams({'type': 'PCA', 'c': 1}),
        'PCA-2': PCAParams({'type': 'PCA', 'c': 2}),
        'PCA-10': PCAParams({'type': 'PCA', 'c': 10}),
        'PCA-20': PCAParams({'type': 'PCA', 'c': 20}),
        'PCA-34': PCAParams({'type': 'PCA', 'c': 34}),
        'SVM-sigmoid': SVMParams({'type': 'SVM', 'C': 1.0, 'kernel': 'sigmoid'}),
        'SVM-sigmoid': SVMParams({'type': 'SVM', 'C': 1.0, 'kernel': 'poly'}),
        'SVM-rbf': SVMParams({'type': 'SVM', 'C': 1.0, 'kernel': 'rbf'}),
        'SVM-linear': SVMParams({'type': 'SVM', 'C': 1.0, 'kernel': 'linear'}),
    }

    features_to_test = [
        int(np.sqrt(m)),      # ~6  — clássico RF
        int(m * 0.3),         # ~10
        int(m * 0.5),         # ~17
        int(m * 0.7),         # ~24
        int(m * 0.9),         # ~31
        m,                    # 35  — todas
    ]
    # remove duplicatas mantendo ordem
    features_to_test = list(dict.fromkeys(features_to_test))

    print(f"\nm={m}, variações de featuresPerTree: {features_to_test}")

    for name, sm in splitMethods.items():
        print(f"\n{'='*30} {name} {'='*30}")
        for f in features_to_test:
            forest = JojiForest(
                featuresPerTree=f,
                samplesPerTree=int(n_train * 0.8),
                repeatedSampling=True,
                treeCount=100,
                maxDepth=4,
                gainMethod='gini',
                splitMethod=sm,
                lda_components=2,
                n_jobs=cores-1
            )
            _evaluate(f"Forest[f={f:2d}]", forest, x_train, y_train, x_test, y_test)

if __name__ == '__main__':
    featuresPerTree()

Loaded data.npz

m=34, variações de featuresPerTree: [5, 10, 17, 23, 30, 34]

============================== orthogonal ==============================


Forest[f= 5]                         acc=0.6681  fit= 20.26s  pred= 0.13s
Forest[f=10]                         acc=0.7090  fit= 37.90s  pred= 0.14s
Forest[f=17]                         acc=0.7444  fit= 36.50s  pred= 0.14s
Forest[f=23]                         acc=0.7500  fit= 43.14s  pred= 0.14s
Forest[f=30]                         acc=0.7542  fit= 46.18s  pred= 0.14s
Forest[f=34]                         acc=0.7444  fit= 50.47s  pred= 0.13s

============================== PCA-1 ==============================
Forest[f= 5]                         acc=0.6879  fit=  9.09s  pred= 0.28s
Forest[f=10]                         acc=0.7048  fit= 17.28s  pred= 0.27s
Forest[f=17]                         acc=0.7401  fit= 20.64s  pred= 0.22s
Forest[f=23]                         acc=0.7415  fit= 24.38s  pred= 0.24s
Forest[f=30]                         acc=0.7444  fit= 32.29s  pred= 0.24s
Forest[f=34]                         acc=0.7415  fit= 41.27s  pred= 0.34s

============================== PCA-2 =====

KeyboardInterrupt: 

### C-parameter Selection

In [ ]:
def testCParameter():
    X_train, Y_train, X_test = loadFile('data.npz')
    x_train, x_test, y_train, y_test = splitData(X_train, Y_train)
    n_train, m = x_train.shape

    for c in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]:
        forest = JojiForest(
                    featuresPerTree=25,
                    samplesPerTree=int(n_train * 0.8),
                    repeatedSampling=True,
                    treeCount=20,
                    maxDepth=4,
                    gainMethod='gini',
                    splitMethod=SVMParams({'type': 'SVM', 'C': c, 'kernel': 'sigmoid'}),
                    lda_components=2,
                    n_jobs=cores-1
                )

        forest.fit(x_train, y_train)
        acc, _ = accuracyAndError(forest.predict(x_test), y_test)
        print(f"C={c}: {acc:.4f}")

if __name__ == '__main__':
    testCParameter()

Loaded data.npz
C=0.001: 0.7345
C=0.01: 0.7472
C=0.1: 0.7387
C=1.0: 0.7415
C=10.0: 0.7472
C=100.0: 0.7316
C=1000.0: 0.7331


### Grid Test
- Features per tree
- Samples per tree

In [ ]:
def gridSearch(x_train, y_train, x_test, y_test, splitMethod, treeCount=50, maxDepth=4):
    n_train, m = x_train.shape

    feature_fracs = [0.3, 0.5, 0.7, 0.9, 1.0]
    sample_fracs  = [0.3, 0.5, 0.632, 0.8, 1.0]

    feature_vals = list(dict.fromkeys(max(1, int(m * f)) for f in feature_fracs))
    sample_vals  = list(dict.fromkeys(max(1, int(n_train * f)) for f in sample_fracs))

    results = np.zeros((len(feature_vals), len(sample_vals)))

    print(f"\n{'':>12}", end="")
    for s in sample_vals:
        print(f"  s={s:4d}", end="")
    print()

    for i, f in enumerate(feature_vals):
        print(f"f={f:3d}  ", end="")
        for j, s in enumerate(sample_vals):
            forest = JojiForest(
                featuresPerTree=f,
                samplesPerTree=s,
                repeatedSampling=True,
                treeCount=treeCount,
                maxDepth=maxDepth,
                gainMethod='gini',
                splitMethod=splitMethod,
                n_jobs=cores-1,
                lda_components=2
            )
            forest.fit(x_train, y_train)
            y_hat = forest.predict(x_test)
            acc, _ = accuracyAndError(y_hat, y_test)
            results[i, j] = acc
            print(f"  {acc:.4f}", end="", flush=True)
        print()

    # melhor combinação
    best_i, best_j = np.unravel_index(np.argmax(results), results.shape)
    print(f"\nMelhor: featuresPerTree={feature_vals[best_i]}, "
          f"samplesPerTree={sample_vals[best_j]}, "
          f"acc={results[best_i, best_j]:.4f}")

    return results, feature_vals, sample_vals

if __name__ == '__main__':
    X_train, Y_train, X_test = loadFile('data.npz')
    x_train, x_test, y_train, y_test = splitData(X_train, Y_train)

    gridSearch(
        x_train, y_train, x_test, y_test,
        splitMethod=SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'poly'}),
        treeCount=150,
    )

Loaded data.npz

              s= 636  s=1061  s=1341  s=1698  s=2123
f= 10    0.7232  0.7373  0.7189  0.7189  0.7203
f= 17    0.7076  0.7161  0.7345  0.7401  0.7387
f= 23    0.7133  0.7373  0.7175  0.7260  0.7415
f= 30    0.7048  0.7119  0.7090  0.7133  0.6992
f= 34    0.7105  0.6963  0.7090  0.7006  0.6808

Melhor: featuresPerTree=23, samplesPerTree=2123, acc=0.7415


### Split and Depth Method Selection

In [13]:
def splitDepthMethodSelection():
    X_train, Y_train, X_test = loadFile('data.npz')
    x_train, x_test, y_train, y_test = splitData(X_train, Y_train)
    n_train, m = x_train.shape

    splitMethods = {
        'orthogonal': OrthogonalParams({'type': 'orthogonal'}),
        'PCA-1': PCAParams({'type': 'PCA', 'c': 1}),
        'PCA-2': PCAParams({'type': 'PCA', 'c': 2}),
        'PCA-10': PCAParams({'type': 'PCA', 'c': 10}),
        'PCA-20': PCAParams({'type': 'PCA', 'c': 20}),
        'PCA-34': PCAParams({'type': 'PCA', 'c': 34}),
        'SVM-linear': SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'linear'}),
        'SVM-sigmoid': SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'sigmoid'}),
        'SVM-rbf': SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'rbf'}),
        'SVM-poly': SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'poly'}),
    }

    depth = [2, 3, 4, 5, 8]

    print("\n" + 30 * "=" + " SPLIT METHOD SELECTION " + 30 * "=")

    print("\n--- Single Tree ---")
    for name, sm in splitMethods.items():
        tree = JojiTree(maxDepth=4, gainMethod='gini', splitMethod=sm)
        _evaluate(f"Tree[{name}]", tree, x_train, y_train, x_test, y_test, lda_components=2)

    print("\n" + 30 * "-" + " Forest " + 30 * "-")
    for name, sm in splitMethods.items():
        for d in depth:

            forest = JojiForest(
                featuresPerTree=17,
                samplesPerTree=int(n_train * 0.7),
                repeatedSampling=True,
                treeCount=150,
                maxDepth=d,
                gainMethod='gini',
                splitMethod=sm,
                lda_components=2,
                n_jobs=cores-1
            )
            _evaluate(f"Forest[{name}-{d}]", forest, x_train, y_train, x_test, y_test)

if __name__ == '__main__':
    splitDepthMethodSelection()

Loaded data.npz

============================== SPLIT METHOD SELECTION ==============================

--- Single Tree ---
Tree[orthogonal]                     acc=0.7331  fit=  1.04s  pred= 0.00s
Tree[PCA-1]                          acc=0.7373  fit=  0.52s  pred= 0.00s
Tree[PCA-2]                          acc=0.7401  fit=  1.12s  pred= 0.00s
Tree[PCA-10]                         acc=0.7401  fit=  0.98s  pred= 0.00s
Tree[PCA-20]                         acc=0.7401  fit=  0.95s  pred= 0.00s
Tree[PCA-34]                         acc=0.7401  fit=  0.96s  pred= 0.00s
Tree[SVM-linear]                     acc=0.7203  fit=  1.46s  pred= 0.00s
Tree[SVM-sigmoid]                    acc=0.7274  fit=  2.09s  pred= 0.54s
Tree[SVM-rbf]                        acc=0.7316  fit=  2.60s  pred= 0.78s
Tree[SVM-poly]                       acc=0.7401  fit=  2.72s  pred= 0.79s

------------------------------ Forest ------------------------------
Forest[orthogonal-2]                 acc=0.7331  fit= 32.84s  pred=

Process ForkPoolWorker-804:
Process ForkPoolWorker-806:
Process ForkPoolWorker-808:
Process ForkPoolWorker-805:
Process ForkPoolWorker-803:
Process ForkPoolWorker-807:
Process ForkPoolWorker-802:
Process ForkPoolWorker-801:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.

KeyboardInterrupt: 

## Main Function

In [27]:
def main():
    X_train, Y_train, X_test = loadFile('data.npz')
    x_train, x_test, y_train, y_test = splitData(X_train, Y_train)
    n_train, m = x_train.shape

    splitMethods = {
        # 'orthogonal':  OrthogonalParams({'type': 'orthogonal'}),
        'SVM-linear':  SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'linear'}),
        'SVM-sigmoid': SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'sigmoid'}),
        'SVM-rbf':     SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'rbf'}),
        'SVM-poly':    SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'poly'}),
    }

    lda_options = {
        # 'no-LDA': None,
        # 'LDA-1': 1,
        'LDA-2': 2, # Máximo (#classes - 1)
        # 'LDA-3': 3
    }

    # print("\n" + 30 * "=" + " ÁRVORE ÚNICA " + 30 * "=")
    # for lda_name, lda_c in lda_options.items():
    #     print(f"\n--- {lda_name} ---")
    #     for name, sm in splitMethods.items():
    #         tree = JojiTree(maxDepth=4, gainMethod='gini', splitMethod=sm)
    #         _evaluate(f"Tree[{name}]", tree, x_train, y_train, x_test, y_test,
    #                   lda_components=lda_c)

    # print("\n" + 30 * "=" + " FLORESTA " + 30 * "=")
    # for lda_name, lda_c in lda_options.items():
    #     print(f"\n--- {lda_name} ---")
    #     for name, sm in splitMethods.items():
    #         forest = JojiForest(
    #             featuresPerTree=23,
    #             samplesPerTree=int(n_train * 1),
    #             repeatedSampling=True,
    #             treeCount=500,
    #             maxDepth=4,
    #             gainMethod='gini',
    #             splitMethod=sm,
    #             lda_components=lda_c,
    #         )
    #         _evaluate(f"Forest[{name}]", forest, x_train, y_train, x_test, y_test)

    forest = JojiForest(
                  featuresPerTree=30,
                  samplesPerTree=int(len(X_train) * 0.8),
                  repeatedSampling=True,
                  treeCount=1500,
                  maxDepth=4,
                  gainMethod='gini',
                  splitMethod=OrthogonalParams({'type': 'orthogonal'}),
                  #splitMethod=PCAParams({'type': 'PCA', 'c': 2}),
                  # splitMethod=SVMParams({'type': 'SVM', 'C': 0.1, 'kernel': 'poly'}),
                  lda_components=2,
                  n_jobs=cores-1
             )

    # forest.fit(X_train, Y_train)
    # y_hat = forest.predict(X_test)

    # exportPredictions(y_hat)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    accuracies = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(x_train), start=1):
        X_tr, Y_tr = x_train[train_idx], y_train[train_idx]
        X_val, Y_val = x_train[val_idx], y_train[val_idx]

        forest.samplesPerTree = len(X_tr)

        forest.fit(X_tr, Y_tr)
        y_hat = forest.predict(X_val)
        acc, _ = accuracyAndError(y_hat, Y_val)
        accuracies.append(acc)
        print(f"Fold {fold}: acc={acc:.4f}")

    print(f"Mean acc: {np.mean(accuracies):.4f}  Std: {np.std(accuracies):.4f}")
    # acc, _ = accuracyAndError(y_hat, y_test)
    # print(f"acc={acc:.4f}")

if __name__ == '__main__':
    main()


Loaded data.npz
Fold 1: acc=0.7788
Fold 2: acc=0.7553


Process ForkPoolWorker-1502:
Process ForkPoolWorker-1506:
Process ForkPoolWorker-1503:
Process ForkPoolWorker-1505:
Process ForkPoolWorker-1504:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/li

KeyboardInterrupt: 